# Agent Evaluation — Exact Match and Semantic Scoring
## Did It Get the Right Answer? — Exact Match and Semantic Scoring
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/76-agent-evaluation/agent_evaluation_workbook.ipynb)

Evaluates an LLM agent against a 5-question golden QA set using two metrics: exact match (keyword in answer) and semantic match (cosine similarity ≥ 0.80 between expected and actual embeddings).

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why automated evaluation matters; exact vs semantic match |
| 2 | **Golden Dataset** — 5 questions with expected answers |
| 3 | **run_agent()** — LLM answers each question; compare to expected |
| 4 | **evaluate_answer()** — Exact match + cosine similarity via embeddings |
| 5 | **Full Eval Loop** — Score all 5, aggregate pass rate |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "numpy", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — What Is Agent Evaluation?

### The testing gap

Unit tests verify that code runs correctly. They cannot tell you whether an LLM *answered well*. Agent evaluation fills that gap by measuring output quality against known-good references.

### The golden set pattern

A **golden set** is a hand-curated list of `{question, expected_answer}` pairs. It is your test suite for the agent. A good golden set:

- Covers the representative cases your agent will encounter in production
- Includes hard cases near the decision boundary (paraphrases, edge cases)
- Stays small enough to run cheaply (20–50 rows is typical)

### Evaluation signals

| Signal | What it measures | When to use |
|--------|-----------------|-------------|
| **Exact match** | Substring or string equality | Short factual answers (dates, names, numbers) |
| **Semantic similarity** | Cosine distance in embedding space | Paraphrase-heavy domains |
| **LLM-as-judge** | Model scores quality with reasoning | Open-ended, multi-step, or code answers |

You almost always combine all three — exact match is cheap and crisp, semantic handles rephrasing, LLM-as-judge catches the rest.

> **Prerequisite:** You've used LangGraph `StateGraph` and LangSmith tracing (examples 62, 73).

In [ ]:
import numpy as np
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import END, START, StateGraph

# golden set — hand-curated pairs; these become the regression baseline
GOLDEN_QA_SET = [
    {"question": "What is the capital of France?",                   "expected": "Paris"},
    {"question": "What is 2 + 2?",                                   "expected": "4"},
    {"question": "Who wrote Romeo and Juliet?",                      "expected": "Shakespeare"},
    {"question": "What is the boiling point of water in Celsius?",   "expected": "100"},
    {"question": "What planet is closest to the Sun?",               "expected": "Mercury"},
]

# temperature=0 → deterministic answers; raise for creative tasks
agent_llm   = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

def run_agent(question: str) -> str:
    return agent_llm.invoke(
        [HumanMessage(content=f"Answer briefly in one sentence: {question}")]
    ).content.strip()

def exact_match_score(expected: str, actual: str) -> bool:
    # case-insensitive substring check — works for short factual answers
    return expected.lower() in actual.lower()

print("Golden set loaded:", len(GOLDEN_QA_SET), "rows")
print("Running agent on first question...")
sample = run_agent(GOLDEN_QA_SET[0]["question"])
em = exact_match_score(GOLDEN_QA_SET[0]["expected"], sample)
print(f"  Q: {GOLDEN_QA_SET[0]['question']}")
print(f"  Expected: {GOLDEN_QA_SET[0]['expected']}")
print(f"  Got: {sample}")
print(f"  Exact match: {em}")

## Part 2 — Semantic Scoring with Embeddings

### Why exact match isn't enough

Exact match fails silently on correct paraphrases:

- Expected: `"Shakespeare"` → Got: `"William Shakespeare"` → exact match: **False** ❌ (but semantically correct)
- Expected: `"100"` → Got: `"100 degrees Celsius"` → exact match: **False** ❌

Embedding cosine similarity measures *meaning*, not string equality. Two semantically equivalent phrases land in nearby regions of the embedding space.

### Cosine similarity formula

```
similarity = (A · B) / (‖A‖ · ‖B‖)
```

Range: `[-1, 1]`. In practice, text embeddings cluster between `0.7–1.0` for similar content.

### Threshold calibration

- `≥ 0.90` — near-identical meaning (rephrase of the same sentence)
- `≥ 0.80` — strong semantic match (good default for factual QA)
- `≥ 0.70` — related topic (may pass paraphrases that change meaning slightly)
- `< 0.70` — likely a different answer

Always calibrate thresholds on a labeled validation set — never pick them from intuition alone.

In [ ]:
def semantic_score(expected: str, actual: str, threshold: float = 0.80) -> dict:
    ev = np.array(embed_model.embed_query(expected))
    av = np.array(embed_model.embed_query(actual))
    # cosine similarity: dot product divided by product of magnitudes
    sim = float(np.dot(ev, av) / (np.linalg.norm(ev) * np.linalg.norm(av)))
    return {
        "cosine_similarity": round(sim, 4),
        "semantic_match": sim >= threshold,
        "threshold": threshold,
    }

# --- combined scorer ---
def evaluate_answer(expected: str, actual: str, threshold: float = 0.80) -> dict:
    exact = exact_match_score(expected, actual)
    sem   = semantic_score(expected, actual, threshold)
    score = 1.0 if (exact or sem["semantic_match"]) else sem["cosine_similarity"]
    return {"exact_match": exact, **sem, "score": score}

# --- run full eval loop ---
results = []
print(f"{'Q':<45} {'Exact':>7} {'Sem':>6} {'Sim':>7} {'Score':>7}")
print("-" * 75)
for row in GOLDEN_QA_SET:
    actual = run_agent(row["question"])
    res    = evaluate_answer(row["expected"], actual)
    results.append({**row, "actual": actual, **res})
    print(f"{row['question'][:44]:<45} {str(res['exact_match']):>7} {str(res['semantic_match']):>6} {res['cosine_similarity']:>7.4f} {res['score']:>7.4f}")

pass_rate = sum(1 for r in results if r["score"] == 1.0) / len(results)
avg_sim   = sum(r["cosine_similarity"] for r in results) / len(results)
print(f"\nPass rate: {pass_rate:.0%}  |  Avg cosine: {avg_sim:.3f}")

## Part 3 — LLM-as-Judge

### When embeddings aren't enough

Cosine similarity can't evaluate:
- Whether code compiles and runs correctly
- Whether a multi-step reasoning chain is logically valid
- Whether a generated plan is safe and coherent
- Whether a creative answer is *good* given the instructions

For these, use an LLM to score the response. This is called **LLM-as-judge**.

### Design principles for judge prompts

1. **Separate the judge from the agent** — use a different (often larger) model as the judge
2. **Provide a rubric** — tell the judge exactly what a 0, 1, 2, 3 score means
3. **Request structured output** — JSON with `score` and `reasoning` fields
4. **Include the question and expected answer** — the judge needs full context
5. **Calibrate with labeled examples** — compare judge scores to human labels before trusting them

### Judge as a LangGraph node

Add the judge as a second node in the eval graph. The graph becomes:

```
START → evaluate_node (run agent + basic scores) → judge_node (LLM score) → END
```

This pattern makes it easy to add or swap judge models without touching the agent logic.

In [ ]:
import json as _json
from langchain_core.messages import SystemMessage

judge_llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

JUDGE_SYSTEM = '''You are an impartial answer quality judge.

Given: question, expected answer, actual answer.
Score the actual answer on a 0-2 scale:
  2 = Correct or semantically equivalent to expected
  1 = Partially correct, contains the key fact but adds wrong extras
  0 = Wrong or unrelated

Respond ONLY with JSON: {"score": <0|1|2>, "reasoning": "<one sentence>"}'''

def llm_judge(question: str, expected: str, actual: str) -> dict:
    prompt = f"Question: {question}\nExpected: {expected}\nActual: {actual}"
    raw = judge_llm.invoke([
        SystemMessage(content=JUDGE_SYSTEM),
        HumanMessage(content=prompt),
    ]).content.strip()
    try:
        # parse structured output — LLM was told to return only JSON
        parsed = _json.loads(raw)
        return {"judge_score": parsed["score"], "judge_reasoning": parsed["reasoning"]}
    except Exception:
        return {"judge_score": -1, "judge_reasoning": f"parse error: {raw[:80]}"}

# --- run eval + judge on all rows ---
print(f"{'Q':<40} {'Sim':>7} {'Judge':>7}  Reasoning")
print("-" * 90)
for r in results:
    j = llm_judge(r["question"], r["expected"], r["actual"])
    print(f"{r['question'][:39]:<40} {r['cosine_similarity']:>7.4f} {j['judge_score']:>7}  {j['judge_reasoning'][:45]}")

## Part 4 — LangGraph Eval Pipeline

Wrapping the full evaluation in a LangGraph pipeline makes it:
- Traceable via LangSmith (each row is a separate run)
- Extendable (add retry, re-rank, or logging nodes without rewriting the loop)
- Testable (swap out individual nodes without changing the pipeline)

### State shape

```python
class EvalState(TypedDict):
    question:        str
    expected:        str
    actual:          str        # filled by run_node
    exact_match:     bool       # filled by score_node
    cosine_similarity: float    # filled by score_node
    semantic_match:  bool       # filled by score_node
    score:           float      # filled by score_node
    judge_score:     int        # filled by judge_node
    judge_reasoning: str        # filled by judge_node
```

The graph has three nodes: `run_node → score_node → judge_node`.

In [ ]:
class EvalState(TypedDict):
    question:          str
    expected:          str
    actual:            str
    exact_match:       bool
    cosine_similarity: float
    semantic_match:    bool
    score:             float
    judge_score:       int
    judge_reasoning:   str

def run_node(state: EvalState) -> dict:
    actual = run_agent(state["question"])
    return {"actual": actual}

def score_node(state: EvalState) -> dict:
    res = evaluate_answer(state["expected"], state["actual"])
    return res

def judge_node(state: EvalState) -> dict:
    return llm_judge(state["question"], state["expected"], state["actual"])

g = StateGraph(EvalState)
g.add_node("run_node",   run_node)
g.add_node("score_node", score_node)
g.add_node("judge_node", judge_node)
g.add_edge(START,        "run_node")
g.add_edge("run_node",   "score_node")
g.add_edge("score_node", "judge_node")
g.add_edge("judge_node", END)
eval_app = g.compile()

# run the pipeline over the golden set
pipeline_results = []
for row in GOLDEN_QA_SET:
    init = {
        "question": row["question"], "expected": row["expected"],
        "actual": "", "exact_match": False,
        "cosine_similarity": 0.0, "semantic_match": False,
        "score": 0.0, "judge_score": -1, "judge_reasoning": "",
    }
    r = eval_app.invoke(init)
    pipeline_results.append(r)

# summary
print(f"{'Q':<42} {'Score':>7} {'Judge':>7}")
print("-" * 60)
for r in pipeline_results:
    print(f"{r['question'][:41]:<42} {r['score']:>7.4f} {r['judge_score']:>7}")

avg_score = sum(r["score"] for r in pipeline_results) / len(pipeline_results)
avg_judge = sum(r["judge_score"] for r in pipeline_results if r["judge_score"] >= 0) / len(pipeline_results)
print(f"\nAvg embedding score: {avg_score:.3f}  |  Avg judge score: {avg_judge:.2f}/2")

## Exercises

---

**Exercise 1 — Extend the Golden Set**

Add 3 more question/answer pairs to `GOLDEN_QA_SET` that you expect to be *hard* for the agent (ambiguous phrasing, multi-word answers, or niche facts). Re-run the embedding eval and observe the scores.

---

**Exercise 2 — Threshold Sensitivity**

Modify `evaluate_answer()` to accept a `threshold` parameter. Run the full eval loop twice: once with `threshold=0.70` and once with `threshold=0.90`. Print how many questions pass at each threshold.

Which questions flip from fail → pass when you lower the threshold? Is that a good thing?

---

**Exercise 3 — 5-Point Judge Scale**

Rewrite `JUDGE_SYSTEM` so the judge scores on a 1–5 scale (1 = completely wrong, 5 = perfect) with a one-sentence rubric for each level. Run the pipeline and compare 5-point judge scores to the binary embedding pass/fail.

*Hint:* Update the parsing in `llm_judge()` to read `score` as an integer 1–5.

---

**Exercise 4 — Regression Detection**

Run the full pipeline twice and store both result sets. Write a function `detect_regressions(run1, run2)` that prints any question where `score` dropped by more than 0.1 between runs.

This simulates what happens when you update your prompt or model — you want to catch regressions automatically.

In [ ]:
# Exercise 2 — Threshold Sensitivity
import asyncio

def run_with_threshold(threshold: float):
    passed = 0
    for row in GOLDEN_QA_SET:
        actual = run_agent(row["question"])
        res    = evaluate_answer(row["expected"], actual, threshold=threshold)
        if res["score"] == 1.0:
            passed += 1
    return passed

print("Threshold sensitivity analysis:")
for t in [0.70, 0.80, 0.90]:
    p = run_with_threshold(t)
    print(f"  threshold={t:.2f} → {p}/{len(GOLDEN_QA_SET)} pass ({p/len(GOLDEN_QA_SET):.0%})")

---
## Answer Key

Attempt the exercises before reading below.

In [ ]:
# Answer 3 — 5-Point Judge Scale
JUDGE_5PT = '''You are an answer quality judge. Score the actual answer 1-5:
  5 = Perfect match or semantically equivalent
  4 = Correct but with minor extra or missing detail
  3 = Partially correct — key fact present but significant errors
  2 = Marginally related — touches the topic but mostly wrong
  1 = Completely wrong or unrelated

Respond ONLY with JSON: {"score": <1-5>, "reasoning": "<one sentence>"}'''

def llm_judge_5pt(question: str, expected: str, actual: str) -> dict:
    prompt = f"Question: {question}\nExpected: {expected}\nActual: {actual}"
    raw = judge_llm.invoke([
        SystemMessage(content=JUDGE_5PT),
        HumanMessage(content=prompt),
    ]).content.strip()
    try:
        parsed = _json.loads(raw)
        return {"judge_score_5pt": int(parsed["score"]), "judge_reasoning": parsed["reasoning"]}
    except Exception:
        return {"judge_score_5pt": -1, "judge_reasoning": f"parse error: {raw[:80]}"}

print(f"{'Q':<42} {'Sim':>7} {'J/5':>5}  Reasoning")
print("-" * 85)
for r in results:
    j = llm_judge_5pt(r["question"], r["expected"], r["actual"])
    print(f"{r['question'][:41]:<42} {r['cosine_similarity']:>7.4f} {j['judge_score_5pt']:>5}  {j['judge_reasoning'][:40]}")

In [ ]:
# Answer 4 — Regression Detection
import copy

def run_full_eval():
    run_results = []
    for row in GOLDEN_QA_SET:
        actual = run_agent(row["question"])
        res    = evaluate_answer(row["expected"], actual)
        run_results.append({"question": row["question"], **res})
    return run_results

def detect_regressions(run1: list, run2: list, threshold: float = 0.1) -> list:
    regressions = []
    for r1, r2 in zip(run1, run2):
        drop = r1["score"] - r2["score"]
        if drop > threshold:
            regressions.append({
                "question": r1["question"],
                "run1_score": r1["score"],
                "run2_score": r2["score"],
                "drop": round(drop, 4),
            })
    return regressions

print("Running eval twice to simulate a model update...")
run1 = run_full_eval()
run2 = run_full_eval()  # identical model here — in practice swap model/prompt between runs

regressions = detect_regressions(run1, run2)
if regressions:
    print(f"\n{len(regressions)} regression(s) detected:")
    for reg in regressions:
        print(f"  {reg['question']}: {reg['run1_score']:.4f} → {reg['run2_score']:.4f} (drop={reg['drop']})")
else:
    print("\nNo regressions detected (scores stable across both runs).")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*